# Model Experiment — RandomForest

End-to-end notebook for **RandomForest** on the IEEE-CIS fraud dataset.
Each section is logged as a separate MLflow run inside the
`RandomForest_Training` experiment.


## 0. Setup

In [ ]:
import os
import sys
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

# repo helpers
sys.path.append("..")
from src.preprocessing import (
    load_raw,
    EmailDomainCleaner,
    TransactionAmtFeats,
    CategoricalEncoder,
    NaNFiller,
    HighNullDropper,
    CorrelatedDropper,
)

import mlflow
from mlflow.models.signature import infer_signature

# point this at your DagsHub repo. on kaggle I just set the env vars in
# the notebook UI and skip the dagshub.init() call.
try:
    import dagshub
    dagshub.init(
        repo_owner=os.environ.get("DAGSHUB_USER", "gurtm23"),
        repo_name=os.environ.get("DAGSHUB_REPO", "Fraud-Detection"),
        mlflow=True,
    )
except Exception as e:
    print("dagshub init skipped:", e)
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "file:./mlruns"))

SEED = 42
np.random.seed(SEED)

from sklearn.ensemble import RandomForestClassifier

In [ ]:
EXPERIMENT = "RandomForest_Training"
mlflow.set_experiment(EXPERIMENT)

## 1. Cleaning

Drop columns with too many NaNs, downcast numeric dtypes, and clean up email domains. Logged as the `*_Cleaning` MLflow run.

In [ ]:
# on kaggle:  train, test = load_raw('/kaggle/input/ieee-fraud-detection')
DATA_PATH = os.environ.get("IEEE_DATA", "../data")
train, test = load_raw(DATA_PATH)
print("train:", train.shape, "  test:", test.shape)
print("fraud rate:", train["isFraud"].mean().round(4))


In [ ]:
# quick sanity peek — not exhaustive EDA, just enough to drive cleaning choices
miss = train.isna().mean().sort_values(ascending=False)
print("cols with >90% missing:", (miss > 0.9).sum())
print("cols with >50% missing:", (miss > 0.5).sum())
print("\ntop 10 most-missing columns:")
print(miss.head(10).round(3))


In [ ]:
with mlflow.start_run(run_name="RandomForest_Cleaning"):
    nan_threshold = 0.9
    mlflow.log_param("nan_threshold", nan_threshold)

    dropper = HighNullDropper(threshold=nan_threshold)
    dropper.fit(train.drop(columns=["isFraud"]))
    cleaned_cols = dropper.keep_

    mlflow.log_metric("n_cols_before", train.shape[1] - 1)
    mlflow.log_metric("n_cols_after", len(cleaned_cols))
    print("kept", len(cleaned_cols), "/", train.shape[1] - 1, "cols")


## 2. Feature Engineering

Email-domain bucketing, TransactionAmt log + decimal features, and synthetic time-of-day from `TransactionDT`. All wrapped as sklearn transformers so they go straight into the pipeline.

In [ ]:
with mlflow.start_run(run_name="RandomForest_FeatureEngineering"):
    mlflow.log_param("email_buckets", True)
    mlflow.log_param("amt_log", True)
    mlflow.log_param("amt_decimal", True)
    mlflow.log_param("tx_hour_day", True)

    fe = Pipeline([
        ("drop_high_null", HighNullDropper(threshold=0.9)),
        ("email", EmailDomainCleaner()),
        ("amt", TransactionAmtFeats()),
        ("cat_enc", CategoricalEncoder(min_count=2)),
        ("fillna", NaNFiller(sentinel=-999)),
    ])

    X = train.drop(columns=["isFraud", "TransactionID"])
    y = train["isFraud"]
    X_fe = fe.fit_transform(X)
    print("after FE:", X_fe.shape)
    mlflow.log_metric("n_features_after_fe", X_fe.shape[1])


## 3. Feature Selection

Tried 3 strategies — kept the one with best CV AUC:

1. **Correlation pruning** (drop one of each pair with |r| > 0.95)
2. **Variance threshold** (drop near-constant)
3. **Model-based importance** (top-k by gain — done after a quick fit)


In [ ]:
from sklearn.feature_selection import VarianceThreshold

with mlflow.start_run(run_name="RandomForest_FeatureSelection"):
    corr_drop = CorrelatedDropper(threshold=0.95)
    X_corr = corr_drop.fit_transform(X_fe)
    mlflow.log_metric("after_corr_drop", X_corr.shape[1])
    mlflow.log_param("corr_threshold", 0.95)

    vt = VarianceThreshold(threshold=0.0)
    X_var = pd.DataFrame(
        vt.fit_transform(X_corr),
        columns=X_corr.columns[vt.get_support()],
        index=X_corr.index,
    )
    mlflow.log_metric("after_var_drop", X_var.shape[1])
    print("final feature count:", X_var.shape[1])


## 4. Training

Random Forest. Tested `n_estimators ∈ {200, 300, 500}` and `max_depth ∈ {8, 12, 16, None}`. Unbounded depth **overfit hard** (train AUC ~0.999, CV ~0.91). Capping depth at 16 closed the gap.

In [ ]:
# build the FULL pipeline (cleaning + FE + FS + model). This is what gets
# saved to MLflow so model_inference.ipynb can run it on raw test data.
pipe = Pipeline([
    ("drop_high_null", HighNullDropper(threshold=0.9)),
    ("email", EmailDomainCleaner()),
    ("amt", TransactionAmtFeats()),
    ("cat_enc", CategoricalEncoder(min_count=2)),
    ("fillna", NaNFiller(sentinel=-999)),
    ("corr_drop", CorrelatedDropper(threshold=0.95)),
    ("model", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)),
])


In [ ]:
# 5-fold stratified CV. logged as a separate run.
with mlflow.start_run(run_name="RandomForest_CrossValidation"):
    params = {"n_estimators": 300, "max_depth": 16, "min_samples_leaf": 20, "n_jobs": -1, "random_state": SEED}
    mlflow.log_params(params)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs = []
    X_full = train.drop(columns=["isFraud", "TransactionID"])
    y_full = train["isFraud"]

    for fold, (tr, va) in enumerate(skf.split(X_full, y_full)):
        pipe.set_params(**{f"model__{k}": v for k, v in params.items()})
        pipe.fit(X_full.iloc[tr], y_full.iloc[tr])
        pred = pipe.predict_proba(X_full.iloc[va])[:, 1]
        auc = roc_auc_score(y_full.iloc[va], pred)
        aucs.append(auc)
        mlflow.log_metric("fold_auc", auc, step=fold)
        print(f"fold {fold}  AUC = {auc:.5f}")

    mlflow.log_metric("cv_mean_auc", np.mean(aucs))
    mlflow.log_metric("cv_std_auc", np.std(aucs))
    print("CV mean AUC:", np.mean(aucs))


In [ ]:
# refit on full train and log the pipeline. this is the artifact
# model_inference.ipynb will pull from the registry.
with mlflow.start_run(run_name="RandomForest_FinalFit") as run:
    mlflow.log_params(params)

    Xtr, Xva, ytr, yva = train_test_split(
        X_full, y_full, test_size=0.15, stratify=y_full, random_state=SEED,
    )
    pipe.fit(Xtr, ytr)
    val_pred = pipe.predict_proba(Xva)[:, 1]
    val_auc = roc_auc_score(yva, val_pred)
    mlflow.log_metric("val_auc", val_auc)
    print("hold-out AUC:", val_auc)

    # full refit before logging
    pipe.fit(X_full, y_full)

    sig = infer_signature(X_full.head(5), pipe.predict_proba(X_full.head(5)))
    info = mlflow.sklearn.log_model(pipe, name="pipeline", signature=sig)
    print("logged run:", run.info.run_id)
    print("model_uri:", info.model_uri)    # save this for registration


## 5. Notes

- Anything above ~0.94 CV AUC tends to **overfit** on this dataset (public LB drops a lot).
- Anything below ~0.88 CV AUC is **underfitting** — likely too aggressive feature drop or weak hyperparams.
- The cleaning + FE pipeline is identical across models so the comparison is fair.